In [1]:
!pip install kagglehub tqdm scikit-learn torchvision

import os
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
import kagglehub

root = kagglehub.dataset_download("abhishekbuddiga06/embryo-dataset")

img_dir = os.path.join(root, "embryo_dataset", "embryo_dataset")
ann_dir = os.path.join(root, "embryo_dataset_annotations", "embryo_dataset_annotations")

print("Dataset Loaded")

Using Colab cache for faster access to the 'embryo-dataset' dataset.
Dataset Loaded


In [3]:
phases = [
    'tPB2','tPNa','tPNf','t2','t3','t4','t5','t6','t7','t8',
    't9+','tM','tSB','tB','tEB','tHB'
]

label_map = {p:i for i,p in enumerate(phases)}

In [4]:
def build_dataframe(limit=20000):

    data = []

    for file in tqdm(os.listdir(ann_dir)):

        if not file.endswith(".csv"):
            continue

        emb = file.replace("_phases.csv","")

        ann = pd.read_csv(os.path.join(ann_dir,file), header=None)
        ann.columns = ["phase","start","end"]

        emb_path = os.path.join(img_dir, emb)

        if not os.path.exists(emb_path):
            continue


        f0_path = os.path.join(emb_path, "F0")

        if not os.path.exists(f0_path):
            continue

        images = sorted(os.listdir(f0_path))

        for _, row in ann.iterrows():

            label = label_map[row["phase"]]

            for i in range(row["start"], row["end"], 6):

                if i < len(images):

                    img_path = os.path.join(f0_path, images[i])


                    if not img_path.lower().endswith((".png",".jpg",".jpeg")):
                        continue

                    data.append([img_path, label, emb])

                if len(data) >= limit:
                    return pd.DataFrame(data, columns=["path","label","emb"])

    return pd.DataFrame(data, columns=["path","label","emb"])

In [5]:
df = build_dataframe()

print("Total samples:", len(df))

100%|██████████| 704/704 [00:34<00:00, 20.13it/s]

Total samples: 15270


In [6]:
embryos = df["emb"].unique()

train_ids, temp_ids = train_test_split(embryos, test_size=0.3, random_state=42)
val_ids, test_ids   = train_test_split(temp_ids, test_size=0.5, random_state=42)

train_df = df[df.emb.isin(train_ids)]
val_df   = df[df.emb.isin(val_ids)]
test_df  = df[df.emb.isin(test_ids)]

print("\nSplit sizes:")
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))


Split sizes:
Train: 10682
Val: 2247
Test: 2341


In [7]:
print("\nBefore merge:")
print(train_df["label"].value_counts().sort_index())


train_df.loc[train_df["label"] == 15, "label"] = 14
val_df.loc[val_df["label"] == 15, "label"] = 14
test_df.loc[test_df["label"] == 15, "label"] = 14

print("\nAfter merge:")
print(train_df["label"].value_counts().sort_index())


Before merge:
label
0      349
1     1508
2      286
3     1055
4      182
5     1049
6      311
7      373
8      362
9     1121
10    1744
11     670
12     611
13     427
14     634
Name: count, dtype: int64

After merge:
label
0      349
1     1508
2      286
3     1055
4      182
5     1049
6      311
7      373
8      362
9     1121
10    1744
11     670
12     611
13     427
14     634
Name: count, dtype: int64


In [8]:
num_classes = 15

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(num_classes),
    y=train_df["label"]
)

print("\nClass weights:")
for i, w in enumerate(weights):
    print(f"Class {i}: {w:.4f}")

weights = torch.tensor(weights, dtype=torch.float32).to(device)


Class weights:
Class 0: 2.0405
Class 1: 0.4722
Class 2: 2.4900
Class 3: 0.6750
Class 4: 3.9128
Class 5: 0.6789
Class 6: 2.2898
Class 7: 1.9092
Class 8: 1.9672
Class 9: 0.6353
Class 10: 0.4083
Class 11: 1.0629
Class 12: 1.1655
Class 13: 1.6678
Class 14: 1.1232


In [9]:
class EmbryoDataset(Dataset):

    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        for _ in range(3):
            try:
                row = self.df.iloc[idx]

                img = Image.open(row["path"]).convert("RGB")

                if self.transform:
                    img = self.transform(img)

                label = torch.tensor(row["label"], dtype=torch.long)

                return img, label

            except:
                idx = np.random.randint(0, len(self.df))

        return torch.zeros(3,299,299), torch.tensor(0)

In [10]:
train_tf = transforms.Compose([
    transforms.Resize((320,320)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.CenterCrop(299),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

test_tf = transforms.Compose([
    transforms.Resize((320,320)),
    transforms.CenterCrop(299),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [11]:
train_loader = DataLoader(
    EmbryoDataset(train_df, train_tf),
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    EmbryoDataset(val_df, test_tf),
    batch_size=32
)

test_loader = DataLoader(
    EmbryoDataset(test_df, test_tf),
    batch_size=32
)

In [12]:
import torch.nn as nn
import torch.nn.functional as F

class DistanceAwareLoss(nn.Module):

    def __init__(self, weights, num_classes):
        super().__init__()

        self.ce = nn.CrossEntropyLoss(weight=weights)
        self.num_classes = num_classes

    def forward(self, logits, targets):


        ce_loss = self.ce(logits, targets)


        probs = F.softmax(logits, dim=1)


        class_ids = torch.arange(self.num_classes).float().to(logits.device)
        soft_pred = torch.sum(probs * class_ids, dim=1)


        distance = torch.abs(soft_pred - targets.float())
        distance_penalty = torch.mean(distance / (self.num_classes - 1))


        return ce_loss + 0.4 * distance_penalty

In [13]:
import torchvision.models as models

def get_model(name):

    if name == "mobilenet":
        model = models.mobilenet_v2(weights="DEFAULT")
        model.classifier[1] = nn.Linear(model.last_channel, num_classes)

    elif name == "vgg16":
        model = models.vgg16(weights="DEFAULT")
        model.classifier[-1] = nn.Linear(4096, num_classes)

    elif name == "vgg19":
        model = models.vgg19(weights="DEFAULT")
        model.classifier[-1] = nn.Linear(4096, num_classes)

    elif name == "inception":
        model = models.inception_v3(weights="DEFAULT", aux_logits=True)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, num_classes)

    return model.to(device)

In [14]:
def train_epoch(model, loader, loss_fn, optimizer):

    model.train()
    total_loss = 0

    for x, y in loader:

        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        outputs = model(x)


        if isinstance(outputs, tuple):
            main_out, aux_out = outputs
            loss = loss_fn(main_out, y) + 0.3 * loss_fn(aux_out, y)
        else:
            loss = loss_fn(outputs, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [15]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate(model, loader):

    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for x, y in loader:

            x = x.to(device)
            outputs = model(x)

            if isinstance(outputs, tuple):
                outputs = outputs[0]

            pred = torch.argmax(outputs, dim=1).cpu().numpy()

            preds.extend(pred)
            targets.extend(y.numpy())

    acc = accuracy_score(targets, preds)
    f1 = f1_score(targets, preds, average="weighted")

    return acc, f1

In [16]:
models_list = ["mobilenet", "vgg16", "vgg19", "inception"]

results = {}

for model_name in models_list:

    print("\n==============================")
    print(f" MODEL: {model_name.upper()}")
    print("==============================")

    results[model_name] = {}


    print("\n Training with CrossEntropy")

    model = get_model(model_name)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)

    ce_loss = nn.CrossEntropyLoss(weight=weights)

    for epoch in range(4):

        loss = train_epoch(model, train_loader, ce_loss, optimizer)
        val_acc, val_f1 = evaluate(model, val_loader)

        print(f"""
Epoch {epoch+1}
Train Loss : {loss:.4f}
Val Acc    : {val_acc:.4f}
Val F1     : {val_f1:.4f}
-------------------------------
""")

    test_acc, test_f1 = evaluate(model, test_loader)

    print(f"FINAL CE → Acc={test_acc:.4f}, F1={test_f1:.4f}")

    results[model_name]["CE"] = (test_acc, test_f1)


    print("\n Training with Custom Loss")

    model = get_model(model_name)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)

    custom_loss = DistanceAwareLoss(weights, num_classes)

    for epoch in range(4):

        loss = train_epoch(model, train_loader, custom_loss, optimizer)
        val_acc, val_f1 = evaluate(model, val_loader)

        print(f"""
Epoch {epoch+1}
Train Loss : {loss:.4f}
Val Acc    : {val_acc:.4f}
Val F1     : {val_f1:.4f}
-------------------------------
""")

    test_acc, test_f1 = evaluate(model, test_loader)

    print(f" FINAL CUSTOM → Acc={test_acc:.4f}, F1={test_f1:.4f}")

    results[model_name]["Custom"] = (test_acc, test_f1)


 MODEL: MOBILENET

 Training with CrossEntropy
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 138MB/s]



Epoch 1
Train Loss : 2.6088
Val Acc    : 0.2537
Val F1     : 0.2260
-------------------------------


Epoch 2
Train Loss : 2.2039
Val Acc    : 0.2577
Val F1     : 0.2564
-------------------------------


Epoch 3
Train Loss : 1.9827
Val Acc    : 0.2755
Val F1     : 0.2826
-------------------------------


Epoch 4
Train Loss : 1.8613
Val Acc    : 0.2510
Val F1     : 0.2589
-------------------------------

FINAL CE → Acc=0.2580, F1=0.2531

 Training with Custom Loss

Epoch 1
Train Loss : 2.6844
Val Acc    : 0.2167
Val F1     : 0.2113
-------------------------------


Epoch 2
Train Loss : 2.2346
Val Acc    : 0.2773
Val F1     : 0.2901
-------------------------------


Epoch 3
Train Loss : 2.0257
Val Acc    : 0.2563
Val F1     : 0.2682
-------------------------------


Epoch 4
Train Loss : 1.9004
Val Acc    : 0.2577
Val F1     : 0.2623
-------------------------------

 FINAL CUSTOM → Acc=0.2922, F1=0.2965

 MODEL: VGG16

 Training with CrossEntropy
Downloading: "https://download.pytorch.or

100%|██████████| 528M/528M [00:07<00:00, 77.0MB/s]



Epoch 1
Train Loss : 2.2561
Val Acc    : 0.2394
Val F1     : 0.2401
-------------------------------


Epoch 2
Train Loss : 1.8996
Val Acc    : 0.2813
Val F1     : 0.2870
-------------------------------


Epoch 3
Train Loss : 1.6960
Val Acc    : 0.2216
Val F1     : 0.2298
-------------------------------


Epoch 4
Train Loss : 1.5151
Val Acc    : 0.2385
Val F1     : 0.2486
-------------------------------

FINAL CE → Acc=0.2520, F1=0.2583

 Training with Custom Loss

Epoch 1
Train Loss : 2.3630
Val Acc    : 0.1891
Val F1     : 0.1899
-------------------------------


Epoch 2
Train Loss : 1.9698
Val Acc    : 0.2479
Val F1     : 0.2626
-------------------------------


Epoch 3
Train Loss : 1.7479
Val Acc    : 0.2679
Val F1     : 0.2776
-------------------------------


Epoch 4
Train Loss : 1.5576
Val Acc    : 0.2964
Val F1     : 0.2950
-------------------------------

 FINAL CUSTOM → Acc=0.2717, F1=0.2686

 MODEL: VGG19

 Training with CrossEntropy
Downloading: "https://download.pytorch.or

100%|██████████| 548M/548M [00:09<00:00, 59.2MB/s]



Epoch 1
Train Loss : 2.2929
Val Acc    : 0.1874
Val F1     : 0.1529
-------------------------------


Epoch 2
Train Loss : 1.9295
Val Acc    : 0.1664
Val F1     : 0.1510
-------------------------------


Epoch 3
Train Loss : 1.7105
Val Acc    : 0.2546
Val F1     : 0.2558
-------------------------------


Epoch 4
Train Loss : 1.5482
Val Acc    : 0.2163
Val F1     : 0.2072
-------------------------------

FINAL CE → Acc=0.2482, F1=0.2347

 Training with Custom Loss

Epoch 1
Train Loss : 2.3420
Val Acc    : 0.1940
Val F1     : 0.1988
-------------------------------


Epoch 2
Train Loss : 1.9601
Val Acc    : 0.2234
Val F1     : 0.2215
-------------------------------


Epoch 3
Train Loss : 1.7447
Val Acc    : 0.2301
Val F1     : 0.2409
-------------------------------


Epoch 4
Train Loss : 1.5739
Val Acc    : 0.3071
Val F1     : 0.3048
-------------------------------

 FINAL CUSTOM → Acc=0.2883, F1=0.2825

 MODEL: INCEPTION

 Training with CrossEntropy
Downloading: "https://download.pytorc

100%|██████████| 104M/104M [00:00<00:00, 181MB/s] 



Epoch 1
Train Loss : 2.9608
Val Acc    : 0.2746
Val F1     : 0.2889
-------------------------------


Epoch 2
Train Loss : 2.3374
Val Acc    : 0.2532
Val F1     : 0.2563
-------------------------------


Epoch 3
Train Loss : 2.0113
Val Acc    : 0.2853
Val F1     : 0.2958
-------------------------------


Epoch 4
Train Loss : 1.7663
Val Acc    : 0.2706
Val F1     : 0.2782
-------------------------------

FINAL CE → Acc=0.2738, F1=0.2757

 Training with Custom Loss

Epoch 1
Train Loss : 3.0608
Val Acc    : 0.2394
Val F1     : 0.2538
-------------------------------


Epoch 2
Train Loss : 2.3917
Val Acc    : 0.2639
Val F1     : 0.2675
-------------------------------


Epoch 3
Train Loss : 2.0847
Val Acc    : 0.2826
Val F1     : 0.2924
-------------------------------


Epoch 4
Train Loss : 1.8410
Val Acc    : 0.2844
Val F1     : 0.2927
-------------------------------

 FINAL CUSTOM → Acc=0.2947, F1=0.2998


In [17]:
print("\n FINAL COMPARISON")

for model in results:

    ce = results[model]["CE"]
    cust = results[model]["Custom"]

    print(f"""
{model.upper()}
CE      → Acc={ce[0]:.4f}, F1={ce[1]:.4f}
Custom  → Acc={cust[0]:.4f}, F1={cust[1]:.4f}
----------------------------------------
""")


 FINAL COMPARISON

MOBILENET
CE      → Acc=0.2580, F1=0.2531
Custom  → Acc=0.2922, F1=0.2965
----------------------------------------


VGG16
CE      → Acc=0.2520, F1=0.2583
Custom  → Acc=0.2717, F1=0.2686
----------------------------------------


VGG19
CE      → Acc=0.2482, F1=0.2347
Custom  → Acc=0.2883, F1=0.2825
----------------------------------------


INCEPTION
CE      → Acc=0.2738, F1=0.2757
Custom  → Acc=0.2947, F1=0.2998
----------------------------------------

